# 1. Box Domain Demo

`BoxDomain` is the concrete 3D rectangular-box geometry in **pygeoinf3D**,
analogous to `IntervalDomain` from the **intervalinf** package.

It provides:
- Volume/extent/center properties
- Containment testing
- Structured (tensor-product) meshes and quadrature weights
- Numerical volume integration
- Uniform interior sampling
- Six labelled face boundaries

All APIs demonstrated here use the **closed** box $[\\mathrm{low}, \\mathrm{high}]\\subset\\mathbb{R}^3$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 – registers 3D projection

from pygeoinf3d import BoxDomain

## 2. Creating `BoxDomain` Variants

Three boxes are constructed to show the flexibility of the API.

In [ ]:
# Unit cube [0,1]^3
unit_cube = BoxDomain(np.zeros(3), np.ones(3))

# Shifted box [1,3] x [0,2] x [-1,1]
shifted_box = BoxDomain(np.array([1.0, 0.0, -1.0]), np.array([3.0, 2.0, 1.0]))

# Thin slab [0,1] x [0,1] x [0,0.1]
thin_slab = BoxDomain(np.zeros(3), np.array([1.0, 1.0, 0.1]))

for name, box in [("unit_cube", unit_cube), ("shifted_box", shifted_box), ("thin_slab", thin_slab)]:
    bb = box.bounding_box()
    print(f"{name}:")
    print(f"  low    = {bb.low}")
    print(f"  high   = {bb.high}")
    print(f"  volume = {box.volume:.6g}")
    print()

## 3. Domain Properties

`BoxDomain` exposes `volume`, `extents`, `center`, and `dim` as properties.

In [ ]:
box = unit_cube

print("=== Unit cube properties ===")
print(f"volume  : {box.volume}")
print(f"extents : {box.extents}")
print(f"center  : {box.center}")
print(f"dim     : {box.dim}")

print()
print("=== Shifted box volume check ===")
expected = 2.0 * 2.0 * 2.0   # (3-1) * (2-0) * (1-(-1))
print(f"volume = {shifted_box.volume:.6g}  (expected {expected:.6g})")
assert abs(shifted_box.volume - expected) < 1e-12, "Volume mismatch!"
print("Volume check PASSED")

## 4. Containment Testing

`contains(point)` tests a single `(3,)` point against the closed box.
Corner and boundary points are included (closed-set semantics).

In [ ]:
test_points = np.array([
    [0.5,  0.5,  0.5],   # interior
    [0.0,  0.0,  0.0],   # corner (low)
    [1.0,  1.0,  1.0],   # corner (high)
    [1.5,  0.5,  0.5],   # outside (+x)
    [-0.1, 0.5,  0.5],   # outside (-x)
])
labels = ["interior", "corner(low)", "corner(high)", "outside(+x)", "outside(-x)"]

print(f"{'Point':<25} {'contains':>8}  Label")
print("-" * 50)
for pt, label in zip(test_points, labels):
    result = unit_cube.contains(pt)
    print(f"{str(pt):<25} {str(result):>8}  {label}")

## 5. Volume Integration

`integrate_volume(f, resolution=n)` uses tensor-product midpoint (cell-centre)
quadrature.  The callable **must be vectorised**: `f(pts:(N,3)) → (N,)` array.

In [ ]:
examples = [
    ("∫1 dx",               lambda x: np.ones(len(x)),                        1.0),
    ("∫x₁ dx",              lambda x: x[:, 0],                                0.5),
    ("∫x₁x₂x₃ dx",         lambda x: x[:, 0] * x[:, 1] * x[:, 2],          0.125),
    ("∫sin(πx₁) dx",        lambda x: np.sin(np.pi * x[:, 0]),               2.0 / np.pi),
    ("∫(x₁²+x₂²+x₃²) dx",  lambda x: np.sum(x**2, axis=1),                  1.0),
]

print(f"{'Integral':<25} {'Numeric':>12} {'Analytic':>12} {'|Error|':>12}")
print("-" * 65)
for name, f, analytic in examples:
    numeric = unit_cube.integrate_volume(f, resolution=30)
    err = abs(numeric - analytic)
    print(f"{name:<25} {numeric:>12.8f} {analytic:>12.8f} {err:>12.2e}")

## 6. Structured Mesh and Volume Weights

`structured_mesh(n)` returns an $(n^3, 3)$ array of cell-centre points.
`structured_mesh(n, location='nodes')` returns an $(n^3, 3)$ array of node points.
`volume_weights(n)` returns matching quadrature weights that sum to the domain volume.

In [ ]:
# Cell-centre mesh: resolution 6 → 6³ = 216 points
pts_centers = unit_cube.structured_mesh(6)
print(f"cell_centers shape : {pts_centers.shape}   (expected (216, 3))")

# Node mesh: resolution 7 → 7³ = 343 nodes (includes boundaries)
pts_nodes = unit_cube.structured_mesh(7, location="nodes")
print(f"nodes shape        : {pts_nodes.shape}   (expected (343, 3))")

# Volume weights
w = unit_cube.volume_weights(6)
print(f"volume_weights sum : {w.sum():.6g}   (expected {unit_cube.volume:.6g})")
print(f"weights shape      : {w.shape}")

# 3-D scatter plot coloured by z
fig = plt.figure(figsize=(6, 5))
ax = fig.add_subplot(111, projection="3d")
sc = ax.scatter(
    pts_centers[:, 0], pts_centers[:, 1], pts_centers[:, 2],
    c=pts_centers[:, 2], cmap="viridis", s=10, alpha=0.6,
)
plt.colorbar(sc, ax=ax, label="z coordinate")
ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")
ax.set_title("Structured mesh — cell centres (6³ = 216)")
plt.tight_layout()
plt.show()

## 7. Interior Sampling

`sample_interior(n, rng=...)` draws $n$ uniform random points from the interior.
The optional `rng=` argument accepts a seed integer or a `np.random.Generator` for reproducibility.

In [ ]:
pts = unit_cube.sample_interior(2000, rng=np.random.default_rng(42))
print(f"shape : {pts.shape}   (expected (2000, 3))")
print(f"x range : [{pts[:,0].min():.4f}, {pts[:,0].max():.4f}]")
print(f"y range : [{pts[:,1].min():.4f}, {pts[:,1].max():.4f}]")
print(f"z range : [{pts[:,2].min():.4f}, {pts[:,2].max():.4f}]")

# Monte-Carlo estimate of ∫x·y·z dx over [0,1]³ — exact value 0.125
mc_est = np.mean(pts[:, 0] * pts[:, 1] * pts[:, 2]) * unit_cube.volume
print(f"\nMonte-Carlo ∫x·y·z dx ≈ {mc_est:.6f}  (exact = 0.125000)")

# 3-D scatter of 400 points coloured by z
fig = plt.figure(figsize=(6, 5))
ax = fig.add_subplot(111, projection="3d")
sub = pts[:400]
sc = ax.scatter(sub[:, 0], sub[:, 1], sub[:, 2],
                c=sub[:, 2], cmap="plasma", s=8, alpha=0.7)
plt.colorbar(sc, ax=ax, label="z coordinate")
ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")
ax.set_title("Interior samples — 400 of 2000 shown")
plt.tight_layout()
plt.show()

## 8. Boundary Faces

`boundary_components` is a `dict` mapping six string keys to `_BoxFaceBoundary` objects.
Each face has a `.contains(point)` method and a `.dim` property (always 2 — a surface).

In [ ]:
bc = unit_cube.boundary_components
print("=== Boundary components ===")
print(f"{'Name':<10} {'Type':<35} {'dim':>4}")
print("-" * 55)
for name, face in bc.items():
    print(f"{name:<10} {type(face).__name__:<35} {face.dim:>4}")

# Containment table for 4 test points
probe_points = {
    "x=0 face centre"  : np.array([0.0, 0.5, 0.5]),
    "x=1 face centre"  : np.array([1.0, 0.5, 0.5]),
    "interior"         : np.array([0.5, 0.5, 0.5]),
    "z=0 face corner"  : np.array([0.5, 0.5, 0.0]),
}

print()
print(f"{'Point':<20}", end="")
for name in bc:
    print(f" {name:>8}", end="")
print()
print("-" * (20 + 9 * len(bc)))
for label, pt in probe_points.items():
    print(f"{label:<20}", end="")
    for face in bc.values():
        result = face.contains(pt)
        print(f" {'T' if result else 'F':>8}", end="")
    print()

## 9. Summary

All `BoxDomain` capabilities demonstrated in this notebook:

| Feature | Method / Property |
|---------|-------------------|
| Construction | `BoxDomain(low, high)` |
| Volume | `box.volume` |
| Side lengths | `box.extents` |
| Midpoint | `box.center` |
| Spatial dimension | `box.dim` |
| Bounding box | `box.bounding_box()` → `AxisAlignedBoundingBox` |
| Point containment | `box.contains(point)` |
| Volume integral | `box.integrate_volume(f, resolution=n)` |
| Cell-centre mesh | `box.structured_mesh(n)` |
| Node mesh | `box.structured_mesh(n, location='nodes')` |
| Quadrature weights | `box.volume_weights(n)` |
| Interior samples | `box.sample_interior(n, rng=...)` |
| Face boundaries | `box.boundary_components` — dict of `_BoxFaceBoundary` |

**Next:** [2_functions_3d_demo.ipynb](2_functions_3d_demo.ipynb)